#### ei hätää. funda muuttujissa käytetään aina kuun viimeistä arvoa. jos ei ole, käytetään sitä edellistä. jos sitäkään ei ole, koko vitun osake pois universumista. tarkista smartestimaattien description, että onko arvioita vai laskettu. 

In [1]:
import refinitiv.data as rd
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

rd.open_session()

<refinitiv.data.session.Definition object at 0x134a24830 {name='workspace'}>

In [2]:
df = rd.get_data(
    universe=["0#.STOXX"],
    fields=["TR.PriceClose"],
)

snapshot = (
    df[["Instrument"]]
    .rename(columns={"Instrument": "RIC"})
    .dropna()
    .drop_duplicates()
    .sort_values("RIC")
    .reset_index(drop=True)
)

snapshot

,RIC
0,A2.MI
1,AAF.L
2,AAK.ST
3,AAL.L
4,AALB.AS
...,...
595,ZAB.WA
596,ZALG.DE
597,ZEG.L
598,ZELA.CO


In [ ]:


universe = snapshot['RIC'].tolist()

fields = [
    "TR.PriceClose.date",
    "TR.PriceClose", # "dirty price" without dividends
    "TR.TotalReturn1D", # dividends included
    "TR.CompanyMarketCap",
    "TR.Volume",
    "TR.SharesOutstanding",
    "TR.TotalDebt",
    "TR.PriceToSalesPerShare",
    "TR.PriceToCFPerShare", 
    "TR.PriceToBVPerShare",
    "TR.GICSSector",
    "TR.GICSSubIndustry",
    "TR.SmartNetDebtToMarketCap",
    "TR.FwdPtoEPSSmartEst",
    "TR.PtoEPSMeanEst",
    "TR.F.CF",
    #"TR.F.DivYldComStockIssuePctAnnized",
    #"TR.F.DivYldComStockGrossIssuePct",
    "TR.DPSActValue",
    #"TR.RelValDividendYieldTTM",
    "TR.F.DivPerShare",
    "TR.DPSActValueYield"
]

params = {
    "SDate": "2024-01-01",
    "EDate": "2026-03-05",
    "Frq": "D",
    "Curn": "EUR"
}

df = rd.get_data(universe=universe, 
                 fields=fields, 
                 parameters=params)


df.head()


In [60]:
df['Div_Yield_Manual'] = (df['Dividend Per Share - Actual'] / df['Price Close']) * 100


In [61]:
df

,Instrument,Date,Price Close,Daily Total Return,Company Market Cap,Volume,Outstanding Shares,Total Debt,Price To Sales Per Share (Daily Time Series Ratio),Price To Cash Flow Per Share (Daily Time Series Ratio),Price To Book Value Per Share (Daily Time Series Ratio),GICS Sector Name,GICS Sub-Industry Name,Net Debt / Market Cap (SmartEstimate),Price / EPS (SmartEstimate ®),Price / EPS (Mean Estimate),Cash Flow,Dividend Per Share - Actual,Dividend Per Share Yield % (Actual),Div_Yield_Manual
0,NESTE.HE,2024-01-02,32.48,0.838249,24983975163.84,953336,768199747,2615000000,1.077639,10.722736,3.085338,Energy,Oil & Gas Refining & Marketing,0.08889,11.321531,11.370797,2527000000,1.52,4.679803,4.679803
1,NESTE.HE,2024-01-03,31.8,-2.093596,24460911644.400002,997501,768199747,2615000000,1.055077,10.498245,3.020743,,,0.09079,11.084504,11.132739,2527000000,1.52,4.779874,4.779874
2,NESTE.HE,2024-01-04,32.27,1.477987,24822440841.66,606915,768199747,2615000000,1.070671,10.653408,3.06539,,,0.089676,11.234234,11.277066,2527000000,1.52,4.710257,4.710257
3,NESTE.HE,2024-01-05,32.4,0.402851,24922438279.199902,729283,768199747,2615000000,1.074984,10.696325,3.077739,,,0.089316,11.279491,11.322495,2527000000,1.52,4.691358,4.691358
4,NESTE.HE,2024-01-08,32.27,-0.401235,24822440841.659901,777006,768199747,2615000000,1.070671,10.653408,3.06539,,,0.090417,11.24641,11.285703,2527000000,1.52,4.710257,4.710257
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,NESTE.HE,2026-02-27,21.18,-0.56338,16291890208.440001,3224564,768274059,5183000000,0.829949,10.330939,2.249917,,,0.226283,18.622072,18.86574,1078000000,0.2,0.944287,0.944287
541,NESTE.HE,2026-03-02,22.65,6.94051,17422630463.700001,3320871,768274059,5183000000,0.887551,11.047958,2.406073,,,0.211597,19.914539,20.175118,1078000000,0.2,0.883002,0.883002
542,NESTE.HE,2026-03-03,22.71,0.264901,17468783127.18,2759665,768274059,5183000000,0.889902,11.077225,2.412446,,,0.211038,19.967293,20.228562,1078000000,0.2,0.880669,0.880669
543,NESTE.HE,2026-03-04,22.53,-0.792602,17330325136.740002,1986712,768274059,5183000000,0.911355,9.920049,2.366587,,,0.211508,20.228775,20.235134,1078000000,0.2,0.887705,0.887705


In [62]:
universe = ['NESTE.HE']

fields = [
    "TR.DividendYield.date",
    "TR.DividendYield"
]

params = {
    "SDate": "2024-01-01",
    "EDate": "2026-03-05",
    "Frq": "D",
    "Curn": "EUR"
}

df_dy = rd.get_data(universe=universe, 
                 fields=fields, 
                 parameters=params)


df_dy.head()


,Instrument,Date,Dividend yield
0,NESTE.HE,2024-01-01,4.701516
1,NESTE.HE,2024-01-02,4.679803
2,NESTE.HE,2024-01-03,4.779874
3,NESTE.HE,2024-01-04,4.710257
4,NESTE.HE,2024-01-05,4.691358


In [63]:
#merge df_dy and df based on date and instrument, keeping all rows from df and matching rows from df_dy where available
df_merged = pd.merge(df, df_dy, left_on=['Date', 'Instrument'], right_on=['Date', 'Instrument'], how='left')

In [65]:
df_merged

,Instrument,Date,Price Close,Daily Total Return,Company Market Cap,Volume,Outstanding Shares,Total Debt,Price To Sales Per Share (Daily Time Series Ratio),Price To Cash Flow Per Share (Daily Time Series Ratio),...,GICS Sector Name,GICS Sub-Industry Name,Net Debt / Market Cap (SmartEstimate),Price / EPS (SmartEstimate ®),Price / EPS (Mean Estimate),Cash Flow,Dividend Per Share - Actual,Dividend Per Share Yield % (Actual),Div_Yield_Manual,Dividend yield
0,NESTE.HE,2024-01-02,32.48,0.838249,24983975163.84,953336,768199747,2615000000,1.077639,10.722736,...,Energy,Oil & Gas Refining & Marketing,0.08889,11.321531,11.370797,2527000000,1.52,4.679803,4.679803,4.679803
1,NESTE.HE,2024-01-03,31.8,-2.093596,24460911644.400002,997501,768199747,2615000000,1.055077,10.498245,...,,,0.09079,11.084504,11.132739,2527000000,1.52,4.779874,4.779874,4.779874
2,NESTE.HE,2024-01-04,32.27,1.477987,24822440841.66,606915,768199747,2615000000,1.070671,10.653408,...,,,0.089676,11.234234,11.277066,2527000000,1.52,4.710257,4.710257,4.710257
3,NESTE.HE,2024-01-05,32.4,0.402851,24922438279.199902,729283,768199747,2615000000,1.074984,10.696325,...,,,0.089316,11.279491,11.322495,2527000000,1.52,4.691358,4.691358,4.691358
4,NESTE.HE,2024-01-05,32.4,0.402851,24922438279.199902,729283,768199747,2615000000,1.074984,10.696325,...,,,0.089316,11.279491,11.322495,2527000000,1.52,4.691358,4.691358,4.691358
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
754,NESTE.HE,2026-02-27,21.18,-0.56338,16291890208.440001,3224564,768274059,5183000000,0.829949,10.330939,...,,,0.226283,18.622072,18.86574,1078000000,0.2,0.944287,0.944287,0.944287
755,NESTE.HE,2026-03-02,22.65,6.94051,17422630463.700001,3320871,768274059,5183000000,0.887551,11.047958,...,,,0.211597,19.914539,20.175118,1078000000,0.2,0.883002,0.883002,0.881834
756,NESTE.HE,2026-03-03,22.71,0.264901,17468783127.18,2759665,768274059,5183000000,0.889902,11.077225,...,,,0.211038,19.967293,20.228562,1078000000,0.2,0.880669,0.880669,0.880669
757,NESTE.HE,2026-03-04,22.53,-0.792602,17330325136.740002,1986712,768274059,5183000000,0.911355,9.920049,...,,,0.211508,20.228775,20.235134,1078000000,0.2,0.887705,0.887705,0.887705


In [64]:
#df['Div_Yield_Manual_notpct'] = (df['Dividend Per Share - Actual'] / df['Price Close'])


In [46]:
print(df.columns.tolist())

['Instrument', 'Date', 'Price Close', 'Daily Total Return', 'Company Market Cap', 'Volume', 'Outstanding Shares', 'Total Debt', 'Price To Sales Per Share (Daily Time Series Ratio)', 'Price To Cash Flow Per Share (Daily Time Series Ratio)', 'Price To Book Value Per Share (Daily Time Series Ratio)', 'GICS Sector Name', 'GICS Sub-Industry Name', 'Net Debt / Market Cap (SmartEstimate)', 'Price / EPS (SmartEstimate ®)', 'Price / EPS (Mean Estimate)', 'Cash Flow', 'Dividend yield', 'Dividend Per Share - Actual', 'Dividend Per Share Yield % (Actual)', 'Div_Yield_Manual']


In [47]:
# df['Div_Yield_Lagged'] = df.groupby('Instrument')['Div_Yield_Manual'].shift(1)

In [48]:
# df_neste = df.loc[
#     df['Instrument'].eq('NESTE.HE'),
#     ['Date', 'Dividend Per Share - Actual']
# ].copy()